# Representative Shapes by Embedding Region

For each embedding method (UMAP, PHATE), each modality (cell, nucleus):
`main.py` fits ONE embedding across every condition and BOTH frames
(1 and 13) together, so this is a single shared space, not two
independent ones. This notebook k-means clusters the FULL embedding -
every cell from both frames pooled together - into a small number of
regions, then picks the one real cell closest to each region's own
centroid and shows its actual shape, cropped from its extracted mask. A
region's representative cell can come from either frame.

2 embedding methods x 2 modalities = 4 plots total.

Requires `main.py` to have already been run.


In [ ]:
import os
import sys

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/03_multi_drug_comparison/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/MASKS_DIR/CAJAL_OUTPUT_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset
from shared.representative_shapes import export_representative_shapes


## Plot: embedding regions and their representative shapes

For each embedding method, one PDF and one SVG (in its own `svgs/`
subfolder), written to `output/plots/00_representative_shapes/`.


In [ ]:
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '00_representative_shapes')

# how many regions to cluster the full embedding into, and how large a
# crop (half-width in pixels) to pull each representative cell's shape
# from - half_width is a guess at "large enough for any cell in this
# dataset"; crop_cell_shape prints a clear warning if a representative
# cell's own shape doesn't fully fit, so bump this up if that happens
N_REGIONS   = 5
HALF_WIDTH  = 150

# main.py computes both of these - plot both every time this notebook
# runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    For each embedding method, load this modality's shared embedding
    (every condition, both frames pooled together) and cluster the
    WHOLE thing into N_REGIONS regions - a region's representative cell
    can come from either frame.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        dfs: {embedding_method: every cell's metadata + dim_0/dim_1,
              plus a 'region' column} (in-memory only)
    '''
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} cells across {df['frame'].nunique()} frames")

        file_prefix = f'{modality}_{embedding_method}'
        df = export_representative_shapes(
            df, config.MASKS_DIR, modality, N_REGIONS, embedding_method.upper(),
            f'{MODALITY_NAMES[modality]} morphology (representative regions)',
            NOTEBOOK_PLOTS_DIR, file_prefix, half_width=HALF_WIDTH,
        )
        dfs[embedding_method] = df

    return dfs


## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc')
